In [1]:
import pandas as pd
import matplotlib.pyplot as plt


# df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df = pd.read_csv('../../datasets/MX_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,5Z75FvRFiultmPFWHx5jQ7,7 Dias,"Gabito Ballesteros, Tito Double P",1,0,1,MX,2025-02-17,85,True,...,-3.749,1,0.0614,0.3860,0.000000,0.0839,0.577,111.913,3,Higher
1,78HEzDEs1QUnHB2DbxgC1s,Te Quería Ver,"Alemán, Neton Vega",2,1,1,MX,2025-02-17,82,False,...,-5.182,0,0.0681,0.1880,0.000017,0.0922,0.448,100.019,4,About_Average
2,0LTwdL5yZ6YOTEGUQPFuSN,ROSONES,Tito Double P,3,1,1,MX,2025-02-17,88,True,...,-5.939,1,0.0318,0.7040,0.000010,0.1170,0.604,120.129,3,Lower
3,7sd6zMrgGpEa7NkQm9TRrg,NADIE,Tito Double P,4,1,2,MX,2025-02-17,87,True,...,-4.710,1,0.1140,0.4650,0.000000,0.1200,0.526,92.604,4,About_Average
4,4eLDmhsJW3JoZTXCAozHor,Loco,Neton Vega,5,-3,45,MX,2025-02-17,61,False,...,-5.502,1,0.0686,0.0741,0.007680,0.1390,0.636,91.981,4,Lower


In [2]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [3]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
import xgboost as xgb

preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    xgb.XGBRegressor(tree_method='hist', max_bin=255)
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("XGBoost")
mean_squared_error(y_test, y_pred)

XGBoost


29.876516342163086

In [4]:
from sklearn.model_selection import GridSearchCV
# set up our search grid
param_grid = {"xgbregressor__max_depth":    [3, 4, 5, 6, 7, 8],
              "xgbregressor__n_estimators": [100, 500],
              "xgbregressor__learning_rate": [0.01, 0.015, 0.02, 0.025, 0.03],
              "xgbregressor__min_child_weight": [1,3,5,10],}

# try out every combination of the above values
search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_squared_error").fit(X_train, y_train)

print("The best hyperparameters are ",search.best_params_)

The best hyperparameters are  {'xgbregressor__learning_rate': 0.025, 'xgbregressor__max_depth': 7, 'xgbregressor__min_child_weight': 3, 'xgbregressor__n_estimators': 500}


In [5]:
search.best_score_

np.float64(-32.55126800537109)

In [6]:
best_pipeline = make_pipeline(
    preprocessing,
    xgb.XGBRegressor(max_depth=search.best_params_["xgbregressor__max_depth"],
                     n_estimators=search.best_params_["xgbregressor__n_estimators"],
                     learning_rate=search.best_params_["xgbregressor__learning_rate"],
                     min_child_weight=search.best_params_["xgbregressor__min_child_weight"],
                     tree_method='hist',
                     max_bin=255,)
)
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
print("XGBoost with best hyperparameters")
print(mean_squared_error(y_test, y_pred))


XGBoost with best hyperparameters
28.738971710205078
